# Подсчёт объектов по целому изображению

Данные соревнования, приватный clevr_counters, две T4, интернет выключен. Контроль — union с фиксированной калибровкой. Первая GPU обучает ResNet, вторая — ConvNeXt-Tiny с подсчётом по областям.

In [ ]:
import hashlib
import json
import os
from pathlib import Path
import shutil
import subprocess
import sys
import tempfile
import time
import zipfile
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import torch
from IPython.display import FileLink, display
from scipy.special import expit
from sklearn.metrics import roc_auc_score

INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")
assert torch.cuda.device_count() == 2, "Выберите GPU T4 x2"
ROOT = Path(tempfile.mkdtemp(prefix="clevr_verify_"))
RESULT = ROOT / "verify_result"
RESULT.mkdir()
print("Рабочая папка", ROOT, flush=True)

def unpack(path, dest):
    with zipfile.ZipFile(path) as z:
        for name in z.namelist():
            if not (dest / name).resolve().is_relative_to(dest.resolve()):
                raise ValueError(name)
        z.extractall(dest)

def find_data(parent):
    return [p.parent for p in parent.rglob("train.csv")
            if all((p.parent / n).is_file() for n in ("val.csv","test.csv","sample_submission.csv"))
            and all((p.parent / n).is_dir() for n in ("train","val","test"))]

locations = list(INPUT.rglob("counters_manifest.json"))
if not locations:
    archives = list(INPUT.rglob("clevr_counters.zip"))
    if len(archives) != 1:
        raise ValueError("Подключите один Private Dataset из clevr_counters.zip")
    unpack(archives[0], ROOT / "package")
    locations = [ROOT / "package/counters_manifest.json"]
if len(locations) != 1:
    raise ValueError("Подключено несколько версий весов")
package = locations[0].parent
for key, digest in json.loads(locations[0].read_text()).items():
    with (package / key).open("rb") as f:
        if hashlib.file_digest(f, "sha256").hexdigest() != digest:
            raise ValueError(key)
datasets = find_data(INPUT)
if not datasets:
    for j, path in enumerate(INPUT.rglob("*.zip")):
        with zipfile.ZipFile(path) as z:
            names = {Path(k).name for k in z.namelist()}
        if {"train.csv","val.csv","test.csv","sample_submission.csv"} <= names:
            unpack(path, ROOT / "inputs" / str(j))
    datasets = find_data(ROOT / "inputs")
if len(datasets) != 1:
    raise ValueError("Подключите данные соревнования")
DATA = datasets[0]
env = dict(os.environ, CLEVR_DATA=str(DATA), CLEVR_PRIORS=str(package/"priors"),
           CLEVR_WEIGHTS=str(package/"weights_in"), PYTHONPATH=str(package),
           PYTHONUNBUFFERED="1", OMP_NUM_THREADS="2", OPENBLAS_NUM_THREADS="2", MKL_NUM_THREADS="2")

## Обучение двух архитектур и прогноз на val

In [ ]:
env.update(FRAGMENT_INPUT=str(package))
def run(arch,seed,stage,gpu):
    folder=ROOT/f"{arch}_{seed}"
    folder.mkdir(exist_ok=True)
    done=folder/f"{stage}.done"
    if done.exists():
        return
    cmd=[sys.executable,"-m","scripts.research.dense_counts",stage,str(seed),"--architecture",arch]
    with (WORK/f"count_{arch}_{seed}_{stage}.log").open("a") as log, subprocess.Popen(cmd,cwd=folder,
            env={**env,"CUDA_VISIBLE_DEVICES":str(gpu)},stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,text=True,bufsize=1) as proc:
        for line in proc.stdout:
            log.write(line)
            print(arch,seed,line,end="",flush=True)
        if proc.wait():
            raise RuntimeError(f"Ошибка {arch}/{stage}. Повторите ячейку в той же сессии.")
    done.touch()
def parallel(jobs):
    with ThreadPoolExecutor(2) as pool:
        pending=[pool.submit(run,*args) for args in jobs]
        for job in pending:
            job.result()
arches=("r50","convnext")
for stage in ("fit","val"):
    parallel([(arch,421,stage,gpu) for gpu,arch in enumerate(arches)])

## Сравнение с текущей моделью

In [ ]:
def read(path,ids):
    with np.load(path) as z:
        assert np.array_equal(z["ids"],ids) and np.isfinite(z["score"]).all()
        return z["score"]

def blend(split,models):
    df=pd.read_csv(DATA/f"{split}.csv")
    anchor=read(package/f"baseline_{split}.npz",df.id)
    predictions=[read(ROOT/f"{arch}_{seed}/verify_result/control_{seed}_{split}.npz",df.id) for arch,seed in models]
    active=df.category.isin(["counting","comparison"])
    return np.where(active,.5*(anchor+np.mean(predictions,axis=0)),anchor)

df=pd.read_csv(DATA/"val.csv")
y=df.label.to_numpy()
base=read(package/"baseline_val.npz",df.id)
assert abs(roc_auc_score(y,base)-.9997986666666666)<1e-9
scenes=pd.factorize(df.image)[0];num=len(np.unique(scenes))
rng=np.random.default_rng(316)
resamples=rng.multinomial(num,np.full(num,1/num),size=1000)[:,scenes]
def metrics(x):
    delta=[roc_auc_score(y,x,sample_weight=w)-roc_auc_score(y,base,sample_weight=w) for w in resamples]
    return dict(auc=float(roc_auc_score(y,x)),wrong=int(((y==1)&(x<0)).sum()),
                tied=int(((y==1)&(x==0)).sum()),lower=float(np.quantile(delta,.025)),
                upper=float(np.quantile(delta,.975)))
candidates={a:blend("val",[(a,421)]) for a in arches}
report={a:metrics(x) for a,x in candidates.items()}
display(pd.DataFrame(report).T)
# Two architecture comparisons: one-sided 2.5% bound per comparison.
eligible=[a for a in arches if report[a]["lower"]>0]
winner=max(eligible,key=lambda a:report[a]["auc"],default=None)
val_rows=df.assign(base=base,**candidates)
val_rows.to_csv(RESULT/"val_rows.csv",index=False)
print("Архитектура для независимого seed",winner)

## Повтор победителя и результаты

In [ ]:
passed=False
if winner:
    run(winner,853,"fit",0)
    run(winner,853,"val",0)
    repeat=blend("val",[(winner,853)])
    result=blend("val",[(winner,421),(winner,853)])
    report["repeat"]=metrics(repeat)
    report["two_seeds"]=metrics(result)
    val_rows.assign(repeat=repeat,two_seeds=result).to_csv(RESULT/"val_rows.csv",index=False)
    passed=report["repeat"]["auc"]>roc_auc_score(y,base) and report["two_seeds"]["lower"]>0
    display(pd.DataFrame(report).T)
if passed:
    parallel([(winner,421,"test",0),(winner,853,"test",1)])
    frame=pd.read_csv(DATA/"test.csv")
    sample=pd.read_csv(DATA/"sample_submission.csv")
    assert frame.id.is_unique and sample.id.is_unique and set(frame.id)==set(sample.id)
    logits=blend("test",[(winner,421),(winner,853)])
    sample["label"]=pd.Series(expit(logits),index=frame.id).loc[sample.id].to_numpy()
    name=f"{winner}_count.csv"
    sample.to_csv(WORK/name,index=False,float_format="%.17g")
    assert sample.equals(pd.read_csv(WORK/name,float_precision="round_trip")) and sample.label.between(0,1).all()
(RESULT/"metrics.json").write_text(json.dumps(dict(report=report,winner=winner,eligible=bool(passed)),indent=2))
with zipfile.ZipFile(WORK/"count_results.zip","w",compression=zipfile.ZIP_DEFLATED,compresslevel=1) as z:
    for folder in [RESULT,*ROOT.glob("*/verify_result")]:
        for path in folder.glob("*"):
            z.write(path,path.relative_to(ROOT))
    for path in WORK.glob("count_*.log"):
        z.write(path,path.name)
os.chdir(WORK)
display(FileLink("count_results.zip"))
if passed:
    display(FileLink(name))